# H1L2 — Implementing attention

**Notebook 2 of this week.** We implement attention for a real sentence in
NumPy, without any framework. The parts follow the slides:

1. tokens become vectors
2. queries, keys and values from learned matrices
3. the attention weights of a sentence
4. the causal mask, for predicting the next token
5. attention as one function
6. several heads
7. attention inside a transformer block
8. attention does not see the order of the tokens
9. adding positions, and the scale

The weights are random: every property shown here holds before any training.

**Colab:** Runtime → Run all. The CPU runtime is enough; this takes seconds.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
C_BLUE, C_RED, C_TEAL, C_PURPLE, C_GRAY = \
    "#1F4E79", "#C00000", "#0E7C7B", "#7030A0", "#595959"
FIGDIR = os.environ.get("SEMINAR_FIGDIR")


def save(fig, name):
    """Only when the slides are built: store the figure for the deck."""
    if FIGDIR:
        for ext in ("pdf", "png"):
            fig.savefig(os.path.join(FIGDIR, f"{name}.{ext}"), bbox_inches="tight", dpi=200)


TOKENS = "the cat sat on the mat".split()
T = len(TOKENS)


def show_matrix(M, title, ax=None, cmap="Blues", vmin=None, vmax=None):
    """A shaded matrix with every value printed in its cell."""
    if ax is None:
        _, ax = plt.subplots(figsize=(0.62 * M.shape[1] + 1.6, 0.58 * M.shape[0] + 1.2))
    ax.imshow(np.where(np.isneginf(M), np.nan, M), cmap=cmap, vmin=vmin, vmax=vmax)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            ax.text(j, i, "-inf" if np.isneginf(v) else f"{v:.2f}",
                    ha="center", va="center", fontsize=8)
    ax.set_xticks(range(M.shape[1]), TOKENS[:M.shape[1]])
    ax.set_yticks(range(M.shape[0]), TOKENS[:M.shape[0]])
    ax.set_title(title, fontsize=10)
    return ax


def rand(m, n):
    """A random m x n matrix, scaled by 1/sqrt(m)."""
    return np.random.randn(m, n) / np.sqrt(m)


def softmax(z):
    z = z - z.max(-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)


## Part 1 — Tokens become vectors

The model looks up each token in an **embedding table**, which holds one
learned vector per word of the vocabulary. We use a random table with vectors
of dimension 8. Note that *the* at position 0 and *the* at position 4 receive
the same vector.


In [ ]:
d = 8                                    # the model dimension
vocab = sorted(set(TOKENS))
n_vocab = len(vocab)
E = np.random.randn(n_vocab, d) * 0.8      # the table
X = np.stack([E[vocab.index(t)]
              for t in TOKENS])            # [T, d]

print("X:", X.shape)
print("rows 0 and 4 equal:", bool((X[0] == X[4]).all()))


## Part 2 — Queries, keys and values

In H1L1 we wrote the query, key and value down by hand. In a model they are
computed from the token vector by three learned matrices:

$$q = x W_Q, \qquad k = x W_K, \qquad v = x W_V .$$

With the tokens as rows of $X$, one matrix product computes the vectors of
all tokens.


In [ ]:
Wq, Wk, Wv = [rand(d, d) for _ in range(3)]
Q, K, V = X @ Wq, X @ Wk, X @ Wv
print("Q, K, V:", Q.shape)


## Part 3 — The attention weights of a sentence

As in H1L1, Part 5: the scores are $QK^\top / \sqrt{d_k}$ and the softmax is
applied to each row. The result is a $T \times T$ matrix: row $i$ says how
much token $i$ takes from each token.


In [ ]:
S = Q @ K.T / np.sqrt(d)
P = softmax(S)
show_matrix(P, "attention weights, no mask (each row sums to 1)", vmin=0, vmax=1)
plt.tight_layout(); save(plt.gcf(), "fig-t-w5-h1l2-p3"); plt.show()
print("rows sum to 1:", np.allclose(P.sum(1), 1))


**Exercise.** Parts 1 to 3 print four facts: the shape of $X$, that rows 0 and 4 of $X$ are equal, the shape of $Q$, $K$ and $V$, and that every row of the weights sums to 1. The weights are random. Which of the four would still hold after training?

<details><summary>Answer</summary>

All four. The shapes do not depend on the values of the weights; every row of a softmax sums to 1; and rows 0 and 4 of $X$ are equal because both are the word *the*, looked up in the same table, trained or not. Training changes which positions receive large weights.

</details>


## Part 4 — The causal mask

A language model is trained to predict the next token at every position of
a text at once. Position $i$ must therefore not read positions $j > i$:
the token it has to predict is among them.

We add $-\infty$ to those scores **before** the softmax. Since
$e^{-\infty} = 0$, the future positions receive a weight of exactly zero, and
the remaining weights of each row still sum to 1.


In [ ]:
neg = np.full((T, T), -np.inf)
mask = np.triu(neg, k=1)       # -inf above the diagonal
P_causal = softmax(S + mask)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
show_matrix(S + mask, "scores + mask", ax=axes[0], cmap="RdBu_r")
show_matrix(P_causal, "weights after the softmax", ax=axes[1], vmin=0, vmax=1)
plt.tight_layout(); save(fig, "fig-t-w5-h1l2-p4"); plt.show()
print("largest future weight:", np.triu(P_causal, k=1).max())
print("row sums:", P_causal.sum(1).round(6))


**Exercise.** Replace `-np.inf` by `-10` and run the cell again. Are the future weights still exactly 0?

<details><summary>Answer</summary>

No. The largest future weight becomes about $1.0 \times 10^{-4}$: small, but not zero. Only $-\infty$ gives a weight of exactly zero, which is what makes the KV cache exact.

</details>


## Part 5 — Attention as one function

Parts 2 to 4 together are a few lines. We also check that computing each
query separately, with only the tokens before it, gives the same result as
the matrix version. The queries do not interact: row $i$ of the output
depends only on tokens $1, \ldots, i$.


In [ ]:
def attention(X, Wq, Wk, Wv):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    S = Q @ K.T / np.sqrt(Wq.shape[1])
    neg = np.full(S.shape, -np.inf)
    P = softmax(S + np.triu(neg, k=1))
    return P @ V, P


def one_at_a_time(X, Wq, Wk, Wv):
    K, V = X @ Wk, X @ Wv
    rows = []
    for i in range(len(X)):                 # only tokens 0..i
        s = (X[i] @ Wq) @ K[: i + 1].T / np.sqrt(Wq.shape[1])
        rows.append(softmax(s) @ V[: i + 1])
    return np.stack(rows)


out, _ = attention(X, Wq, Wk, Wv)
loop = one_at_a_time(X, Wq, Wk, Wv)
print(f"largest difference: {np.abs(out - loop).max():.1e}")

fig, ax = plt.subplots(figsize=(3.8, 3.6))
ax.plot(out.ravel(), loop.ravel(), "o", color=C_BLUE, ms=4)
lim = [out.min(), out.max()]
ax.plot(lim, lim, color=C_GRAY, lw=0.8)
ax.set_xlabel("matrix version"); ax.set_ylabel("one query at a time")
ax.set_title("every output entry, computed both ways", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); save(fig, "fig-t-w5-h1l2-p5"); plt.show()


## Part 6 — Several heads

A model runs several attention computations in parallel, each with its own
matrices of a smaller dimension. Each **head** produces its own weight
matrix, so one layer can attend in several ways at once. The outputs of the
heads are concatenated and multiplied by a further matrix $W_O$.

With $H$ heads of dimension $d_{\text{model}}/H$, the number of parameters is
the same as for one head of full width.


In [ ]:
H, dh = 2, d // 2
def head():                    # Wq, Wk, Wv
    return [rand(d, dh) for _ in range(3)]


heads = [head() for _ in range(H)]
Wo = rand(d, d)

outs = [attention(X, *w)[0]
        for w in heads]
multi = np.concatenate(outs, 1) @ Wo
pats = [attention(X, *w)[1] for w in heads]

fig, axes = plt.subplots(1, H, figsize=(10, 3.8))
for h in range(H):
    show_matrix(pats[h], f"head {h}", ax=axes[h], vmin=0, vmax=1)
plt.tight_layout(); save(fig, "fig-t-w5-h1l2-p6"); plt.show()
print("layer output:", multi.shape)


**Exercise.** Set `H, dh = 4, d // 4`. What is the shape of each head's queries, and of the output of the layer?

<details><summary>Answer</summary>

Each head's queries have shape $(6, 2)$, since $d / H = 2$. The output of the layer is still $(6, 8)$: the four outputs of width 2 are concatenated to width 8 and multiplied by $W_O$.

</details>


## Part 7 — Attention inside a transformer block

The output of attention does not replace the token vectors; it is **added**
to them. The sum then passes through an MLP, whose output is added again.
This running sum is the **residual stream**. A model stacks many such
blocks.


In [ ]:
def mlp(x, W1, W2):
    return np.maximum(0, x @ W1) @ W2                   # one hidden layer, ReLU


W1, W2 = rand(d, 4 * d), rand(4 * d, d)


def block(x):
    outs = [attention(x, *w)[0]
            for w in heads]
    a = np.concatenate(outs, 1) @ Wo
    x = x + a
    return x + mlp(x, W1, W2)


print("block(X):", block(X).shape)
print("block(block(X)):", block(block(X)).shape)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
for ax, M, title in [(axes[0], X, "X, the input"), (axes[1], block(X), "block(X), the output")]:
    ax.imshow(M, cmap="RdBu_r", vmin=-3, vmax=3)
    ax.set_yticks(range(T), TOKENS); ax.set_xlabel("dimension")
    ax.set_title(title, fontsize=10)
plt.tight_layout(); save(fig, "fig-t-w5-h1l2-p7"); plt.show()


## Part 8 — Attention does not see the order of the tokens

The two tokens *the* have the same vector, hence the same query and key. We
look at their scores against the tokens both can see (positions 0 to 4, so
without the mask): the two rows are identical. More generally, if we permute
the input tokens, the output rows are permuted in the same way and are
otherwise unchanged. Attention alone has no information about positions.


In [ ]:
# the rows of the two tokens 'the'
print(S[0, :5].round(2))
print(S[4, :5].round(2))

fig, ax = plt.subplots(figsize=(4.6, 4.0))
show_matrix(S, "scores without positions", ax=ax, cmap="RdBu_r")
for r in (0, 4):
    ax.add_patch(plt.Rectangle((-0.5, r - 0.5), T, 1, fill=False, ec=C_RED, lw=2))
plt.tight_layout(); save(fig, "fig-t-w5-h1l2-p8"); plt.show()


def no_mask(X):
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    return softmax(Q @ K.T / np.sqrt(d)) @ V


perm = np.random.permutation(T)
print("permuted:", np.allclose(no_mask(X[perm]),
                               no_mask(X)[perm]))


**Exercise.** Swap the tokens at positions 1 and 2 (*cat* and *sat*) and compute `no_mask` again. Which rows of the output change, and how?

<details><summary>Answer</summary>

Rows 1 and 2 of the output swap places, and no other row changes. Without positions, attention treats the input as a set of tokens.

</details>


## Part 9 — Adding positions, and the scale

Models add information about positions. Here we add to each token vector a
fixed vector that depends only on the position of the token, and take the
function that makes these vectors as given. Other models, Llama among them,
instead rotate the queries and keys by angles that depend on the position
(RoPE); how position encodings work, and why one is chosen over another, is
the topic of position encodings later. Here we only see the effect: the two
*the* tokens now have different scores.

The division by $\sqrt{d_k}$ in the scores is the other line we take as
given. It keeps the scores in a range where the softmax works well, and we
return to it later.


In [ ]:
def position_vectors(T, d, start=0):
    """Fixed sinusoidal position vectors (taken as given this week)."""
    pos = np.arange(start, start + T)[:, None]
    freq = 10000.0 ** (-np.arange(0, d, 2) / d)
    P = np.zeros((T, d))
    P[:, 0::2] = np.sin(pos * freq)
    P[:, 1::2] = np.cos(pos * freq)
    return P


Xp = X + position_vectors(T, d)
Qp, Kp = Xp @ Wq, Xp @ Wk
S_pos = Qp @ Kp.T / np.sqrt(d)
print(S_pos[0, :5].round(2))
print(S_pos[4, :5].round(2))

fig, axes = plt.subplots(1, 2, figsize=(9, 4.0))
for ax, M, title in [(axes[0], S, "without positions"), (axes[1], S_pos, "with position vectors")]:
    show_matrix(M, title, ax=ax, cmap="RdBu_r")
    for r in (0, 4):
        ax.add_patch(plt.Rectangle((-0.5, r - 0.5), T, 1, fill=False, ec=C_RED, lw=2))
plt.tight_layout(); save(fig, "fig-t-w5-h1l2-p9"); plt.show()


**Exercise.** Shift every position by 10, `position_vectors(T, d, start=10)`, and compute the scores again. Do they change?

<details><summary>Answer</summary>

Yes, by up to 2.89. These position vectors are absolute: the scores depend on where the tokens are, not only on how far apart they are. Whether a model should depend on absolute positions is one of the questions of position encodings, later.

</details>


## What this notebook showed

| part | result |
|---|---|
| tokens | the same word gives the same vector, wherever it is |
| the mask | the weight of every future token is exactly 0, before any training |
| one function | computing the queries one at a time gives the same result |
| heads | each head has its own weight matrix |
| order | without positions, permuting the input only permutes the output |
| positions | with position vectors added, the two *the* tokens are distinguished |
